#Gold Layer

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS dwbi.gold;

###Dim Date

In [0]:
df_gold = spark.table("dwbi.silver.gold_ready")

In [0]:
from pyspark.sql import functions as F

dim_date_stage = df_gold.select(F.col("FL_DATE").alias("full_date")).distinct() \
    .withColumn("day_of_week", F.date_format("full_date", "EEEE")) \
    .withColumn("month", F.date_format("full_date", "MMMM")) \
    .withColumn("month_num", F.month("full_date")) \
    .withColumn("is_weekend", F.dayofweek("full_date").isin([1, 7]))

dim_date_stage.createOrReplaceTempView("dim_date_stage")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dwbi.gold.dim_date (
  date_key BIGINT GENERATED ALWAYS AS IDENTITY,
  full_date DATE,
  day_of_week STRING,
  month STRING,
  month_num INT,
  is_weekend BOOLEAN
);

INSERT INTO dwbi.gold.dim_date (full_date, day_of_week, month, month_num, is_weekend)
SELECT full_date, day_of_week, month, month_num, is_weekend FROM dim_date_stage;

num_affected_rows,num_inserted_rows
31,31


###Dim Career

In [0]:
dim_carrier_stage = df_gold.select(
    F.col("OP_UNIQUE_CARRIER").alias("carrier_code"),
    F.col("carrier_name")
).distinct()

dim_carrier_stage.createOrReplaceTempView("dim_carrier_stage")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dwbi.gold.dim_carrier (
  carrier_key BIGINT GENERATED ALWAYS AS IDENTITY,
  carrier_code STRING,
  carrier_name STRING
);

INSERT INTO dwbi.gold.dim_carrier (carrier_code, carrier_name)
SELECT carrier_code, carrier_name FROM dim_carrier_stage;

num_affected_rows,num_inserted_rows
15,15


###Dim Airport

This is a role playing dimention. Both of the origin_airport_key and dest_airport keys in the fact table will be connected here. 

In [0]:
origin_airports = df_gold.select(
    F.col("origin_iata").alias("airport_code"),
    F.col("origin_airport_name").alias("airport_name"),
    F.col("origin_airport_city").alias("airport_city"),
    F.col("origin_latitude").alias("latitude"),
    F.col("origin_longitude").alias("longitude")
)

dest_airports = df_gold.select(
    F.col("dest_iata").alias("airport_code"),
    F.col("dest_airport_name").alias("airport_name"),
    F.col("dest_airport_city").alias("airport_city"),
    F.col("dest_latitude").alias("latitude"),
    F.col("dest_longitude").alias("longitude")
)

dim_airport_stage = origin_airports.union(dest_airports).distinct()
dim_airport_stage.createOrReplaceTempView("dim_airport_stage")

In [0]:
dupe_check = dim_airport_stage.groupBy("airport_code").count().filter(F.col("count") > 1)
display(dupe_check)

airport_code,count


In [0]:
%sql
CREATE TABLE IF NOT EXISTS dwbi.gold.dim_airport (
  airport_key BIGINT GENERATED ALWAYS AS IDENTITY,
  airport_code STRING,
  airport_name STRING,
  airport_city STRING,
  latitude DOUBLE,
  longitude DOUBLE
);

INSERT INTO dwbi.gold.dim_airport (airport_code, airport_name, airport_city, latitude, longitude)
SELECT airport_code, airport_name, airport_city, latitude, longitude FROM dim_airport_stage;

num_affected_rows,num_inserted_rows
334,334


###Dim Time Bucket

In [0]:
df_gold = df_gold.withColumn(
    "dep_time_bucket",
    F.when((F.col("dep_hour") >= 0) & (F.col("dep_hour") < 6), "Late Night")
     .when((F.col("dep_hour") >= 6) & (F.col("dep_hour") < 12), "Morning")
     .when((F.col("dep_hour") >= 12) & (F.col("dep_hour") < 17), "Afternoon")
     .when((F.col("dep_hour") >= 17) & (F.col("dep_hour") < 21), "Evening")
     .otherwise("Night")
)

dim_time_bucket_stage = df_gold.select("dep_time_bucket").distinct()
dim_time_bucket_stage.createOrReplaceTempView("dim_time_bucket_stage")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dwbi.gold.dim_time_bucket (
  time_bucket_key BIGINT GENERATED ALWAYS AS IDENTITY,
  bucket_name STRING
);

INSERT INTO dwbi.gold.dim_time_bucket (bucket_name)
SELECT dep_time_bucket FROM dim_time_bucket_stage;

num_affected_rows,num_inserted_rows
5,5


##Fact Tables

###Fact Flight Performance

In [0]:
dim_date = spark.table("dwbi.gold.dim_date")
dim_carrier = spark.table("dwbi.gold.dim_carrier")
dim_airport = spark.table("dwbi.gold.dim_airport")
dim_time_bucket = spark.table("dwbi.gold.dim_time_bucket")

origin_dim = dim_airport.select(
    F.col("airport_key").alias("origin_airport_key"),
    F.col("airport_code").alias("origin_code_match")
)
dest_dim = dim_airport.select(
    F.col("airport_key").alias("dest_airport_key"),
    F.col("airport_code").alias("dest_code_match")
)

fact_stage = df_gold \
    .join(dim_date, df_gold.FL_DATE == dim_date.full_date, "left") \
    .join(dim_carrier, df_gold.OP_UNIQUE_CARRIER == dim_carrier.carrier_code, "left") \
    .join(origin_dim, df_gold.origin_iata == F.col("origin_code_match"), "left") \
    .join(dest_dim, df_gold.dest_iata == F.col("dest_code_match"), "left") \
    .join(dim_time_bucket, df_gold.dep_time_bucket == dim_time_bucket.bucket_name, "left") \
    .select(
        F.col("date_key"),
        F.col("carrier_key"),
        F.col("origin_airport_key"),
        F.col("dest_airport_key"),
        F.col("time_bucket_key"),
        F.col("DEP_DELAY").alias("dep_delay_min"),
        F.col("ARR_DELAY").alias("arr_delay_min"),
        F.col("ACTUAL_ELAPSED_TIME").alias("actual_elapsed_time_min"),
        F.col("DISTANCE").alias("distance_miles"),
        F.col("CANCELLED").alias("is_cancelled"),
        F.col("DIVERTED").alias("is_diverted"),
        F.col("is_delay_applicable")
    )

fact_stage.createOrReplaceTempView("fact_stage")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dwbi.gold.fact_flight_performance (
  date_key BIGINT,
  carrier_key BIGINT,
  origin_airport_key BIGINT,
  dest_airport_key BIGINT,
  time_bucket_key BIGINT,
  dep_delay_min DOUBLE,
  arr_delay_min DOUBLE,
  actual_elapsed_time_min DOUBLE,
  distance_miles DOUBLE,
  is_cancelled DOUBLE,
  is_diverted DOUBLE,
  is_delay_applicable BOOLEAN
);

INSERT INTO dwbi.gold.fact_flight_performance
SELECT * FROM fact_stage;

num_affected_rows,num_inserted_rows
547271,547271


In [0]:
print("Gold rows:", df_gold.count())
print("Fact rows:", spark.table("dwbi.gold.fact_flight_performance").count())

fact_check = spark.table("dwbi.gold.fact_flight_performance")
fact_check.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in ["date_key", "carrier_key", "origin_airport_key", "dest_airport_key", "time_bucket_key"]
]).display()

Gold rows: 547271
Fact rows: 547271


date_key,carrier_key,origin_airport_key,dest_airport_key,time_bucket_key
0,0,0,0,0


In [0]:
df_airports_check = spark.table("dwbi.bronze.airports_lookup")
print("Total rows:", df_airports_check.count())
print("Distinct iata codes:", df_airports_check.select("iata").distinct().count())

Total rows: 29307
Distinct iata codes: 7918


In [0]:
df_dim_airport = spark.table("dwbi.gold.dim_airport")
print("dim_airport rows:", df_dim_airport.count())
print("Distinct airport_code:", df_dim_airport.select("airport_code").distinct().count())

dim_airport rows: 334
Distinct airport_code: 334


In [0]:
fact_tbl = spark.table("dwbi.gold.fact_flight_performance")
print("Total rows:", fact_tbl.count())
print("Distinct rows:", fact_tbl.distinct().count())

Total rows: 547271
Distinct rows: 545857


In [0]:
df_date = spark.table("dwbi.gold.dim_date")
print("dim_date rows:", df_date.count(), "| distinct dates:", df_date.select("full_date").distinct().count())

df_carrier = spark.table("dwbi.gold.dim_carrier")
print("dim_carrier rows:", df_carrier.count(), "| distinct codes:", df_carrier.select("carrier_code").distinct().count())

dim_date rows: 31 | distinct dates: 31
dim_carrier rows: 15 | distinct codes: 15


###Exporting as .csv

In [0]:
export_config = {
    "dwbi.gold.dim_date": "dim_date",
    "dwbi.gold.dim_carrier": "dim_carrier",
    "dwbi.gold.dim_airport": "dim_airport",
    "dwbi.gold.dim_time_bucket": "dim_time_bucket",
    "dwbi.gold.fact_flight_performance": "fact_flight_performance"
}

export_path = "/Volumes/dwbi/stage/volume/powerbi_export"

In [0]:
import os
os.makedirs(export_path, exist_ok=True)

for table_name, file_name in export_config.items():
    df = spark.table(table_name)
    pdf = df.toPandas()
    pdf.to_csv(f"{export_path}/{file_name}.csv", index=False)
    print(f"Exported {table_name} -> {file_name}.csv ({len(pdf)} rows)")

Exported dwbi.gold.dim_date -> dim_date.csv (31 rows)
Exported dwbi.gold.dim_carrier -> dim_carrier.csv (15 rows)
Exported dwbi.gold.dim_airport -> dim_airport.csv (334 rows)
Exported dwbi.gold.dim_time_bucket -> dim_time_bucket.csv (5 rows)
Exported dwbi.gold.fact_flight_performance -> fact_flight_performance.csv (547271 rows)
